# PD06 · PQRS completas: agregaciones de alta cardinalidad en Polars CPU y GPU
Curso de Postgrado: BigData, Especialización en Bases de datos · CyberColombia.

**Plantilla individual por desarrollar.** Copia este notebook a `soluciones/`
con el mismo nombre. Completa código, controles e interpretación; no se entrega
una solución ni resultados medidos. La GPU y su kernel se consideran verificados.

Compararás **Polars CPU frente a Polars GPU**, con todos los datos de tres
periodos de Supersalud, sin pandas para las consultas. Resolverás dos ejercicios:

1. Departamentos con más quejas según edad, patología, motivo y canal.
2. Evolución de esas mismas combinaciones mes a mes, en orden cronológico ascendente.

Las claves de texto y el número de grupos aumentan el trabajo de agregación y
ordenación. Debes medirlo: no se garantiza saturar la GPU ni obtener aceleración.
No copies filas para simular más datos. Registra la cardinalidad observada.


## 1. Preparar los tres CSV completos

Desde una terminal Ubuntu/WSL, en la raíz del repositorio y con el entorno activo:

```bash
python kit/00_datos.py --descargar pqrs-completos
python kit/00_datos.py --verificar pqrs-completos
```

Deben quedar en `kit/data/pqrs_completos/`:

| Corte | CSV | Reportes de referencia |
|---|---|---:|
| 2023-II | `PQRD_2023_II.csv` | 716.697 |
| 2024-I | `PQRD_2024_I.csv` | 781.601 |
| 2024-II | `PQRD_2024_II.csv` | 946.468 |

Total: **2.444.766 reportes**, aproximadamente 1,82 GB de CSV. El descargador
conserva archivos correctos y verifica los tres. No son muestras.
No ejecutes esos comandos dentro del reloj del benchmark.


In [3]:
from pathlib import Path
from time import perf_counter
from importlib import metadata
import json, platform, sys
import polars as pl
from polars.testing import assert_frame_equal
from IPython.display import display

REPOSITORIO = None  # Ejemplo: Path('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate')
if REPOSITORIO is None:
    REPOSITORIO = next((p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / 'kit/pqrs_fuentes.json').is_file()), None)
assert REPOSITORIO is not None, 'Indicar la ruta del repositorio'
REPOSITORIO = Path(REPOSITORIO).expanduser().resolve()
KIT = REPOSITORIO / 'kit'
CSV_DIR = KIT / 'data/pqrs_completos'
OUT = KIT / 'salidas/soluciones/PD06'
PARQUET_DIR = OUT / 'parquet'
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
PERIODOS = ['2023_II', '2024_I', '2024_II']
ARCHIVOS_CSV = [CSV_DIR / f'PQRD_{periodo}.csv' for periodo in PERIODOS]
ARCHIVOS_PARQUET = [PARQUET_DIR / f'{periodo}.parquet' for periodo in PERIODOS]
FILAS_ESPERADAS = [716697, 781601, 946468]
TOTAL_ESPERADO = sum(FILAS_ESPERADAS)
REPETICIONES = 3
TOP_N = 10
DISPOSITIVO_CUDA = 0
assert all(p.is_file() for p in ARCHIVOS_CSV), 'Completar descarga y verificación'
assert REPETICIONES >= 3
print('Python:', sys.executable)
print('Polars:', pl.__version__)
print('Entrada:', CSV_DIR)
print('Resultados:', OUT)


Python: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
Polars: 1.35.2
Entrada: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/pqrs_completos
Resultados: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD06


## 2. Leer y convertir con Polars · desarrollo del estudiante

Completa la siguiente celda sin pandas ni funciones del taller que oculten la conversión:

1. Inspecciona cinco filas de cada CSV con `pl.read_csv`. Usa `;`, `,` y `;`,
   respectivamente. Conserva códigos y las 38 columnas originales como texto.
2. Usa `pl.scan_csv` y transforma con expresiones Polars. Añade:
   - `corte_archivo`: corte del CSV de origen.
   - `codigo_departamento`: desde `afec_cod_depto`, dos dígitos; marca inválidos.
   - `edad_anios`: desde `afec_edad`, numérica. Explicita el tratamiento de edades
     vacías, negativas, no numéricas o no finitas; conserva estos reportes en un grupo nulo.
   - `anio` y `mes_num`: números desde `periodo` y `mes`, no desde el nombre semestral.
3. Escribe un Parquet por corte con `sink_parquet`, Zstandard y streaming CPU.
4. Comprueba que están los tres, que conservan sus filas y el contrato de columnas.
5. Al reejecutar, comprueba los Parquet existentes antes de decidir reutilizarlos.

La preparación se hace con Polars CPU; la GPU se utilizará para las agregaciones.
No mezcles este tiempo con los de consulta. Guarda el tiempo de conversión aparte.


In [6]:
# TODO: inspeccionar los tres CSV y desarrollar aquí su conversión con Polars.

# 2.1 Reglas explícitas de lectura y normalización

SEPARADORES = {'2023_II': ';', '2024_I': ',', '2024_II': ';'}

# Códigos DIVIPOLA de los 32 departamentos + Bogotá D. C. Fuera de esta lista → inválido.
DEPARTAMENTOS = {
    '05': 'Antioquia', '08': 'Atlántico', '11': 'Bogotá D. C.', '13': 'Bolívar',
    '15': 'Boyacá', '17': 'Caldas', '18': 'Caquetá', '19': 'Cauca', '20': 'Cesar',
    '23': 'Córdoba', '25': 'Cundinamarca', '27': 'Chocó', '41': 'Huila',
    '44': 'La Guajira', '47': 'Magdalena', '50': 'Meta', '52': 'Nariño',
    '54': 'Norte de Santander', '63': 'Quindío', '66': 'Risaralda', '68': 'Santander',
    '70': 'Sucre', '73': 'Tolima', '76': 'Valle del Cauca', '81': 'Arauca',
    '85': 'Casanare', '86': 'Putumayo', '88': 'San Andrés y Providencia',
    '91': 'Amazonas', '94': 'Guainía', '95': 'Guaviare', '97': 'Vaupés', '99': 'Vichada'}

# Edad: texto → Float64 (no numérica → nulo). Se conserva solo si es finita, >= 0 y
# <= EDAD_MAXIMA; se trunca a años cumplidos (floor) y se guarda como Int16.
# Vacías, negativas, no numéricas, NaN/inf o > EDAD_MAXIMA → edad_anios nula
# (el reporte NO se elimina: queda en el grupo de edad nula).
EDAD_MAXIMA = 120

# Fecha: anio desde `periodo` y mes_num desde `mes`; se aceptan números o nombres en español.
# Si año o mes no son válidos, ambos quedan nulos y fecha_valida = False.
ANIO_MINIMO, ANIO_MAXIMO = 2000, 2100
MESES_TEXTO = {'enero': '1', 'febrero': '2', 'marzo': '3', 'abril': '4', 'mayo': '5',
               'junio': '6', 'julio': '7', 'agosto': '8', 'septiembre': '9',
               'setiembre': '9', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'}
# Coherencia (solo control, no filtra): año y meses esperados por corte.
MESES_CORTE = {'2023_II': (2023, list(range(7, 13))),
               '2024_I': (2024, list(range(1, 7))),
               '2024_II': (2024, list(range(7, 13)))}

# cod_motesp: 2024_I lo publica sin ceros a la izquierda ('10304' frente a '010304').
# Se rellena a 6 dígitos para no partir un mismo motivo en dos grupos; el valor
# recibido queda en cod_motesp_origen.
ANCHO_COD_MOTESP = 6
VERSION_REGLAS = 'PD06-reglas-v1'
COLUMNAS_DERIVADAS = ['corte_archivo', 'cod_motesp_origen', 'codigo_departamento',
                      'departamento_valido', 'edad_anios', 'edad_valida', 'anio',
                      'mes_num', 'fecha_valida', 'fecha_en_corte']


# 2.2 Inspección: cinco filas por CSV, todas las columnas como texto

COLUMNAS_ORIGINALES = None
for periodo, csv in zip(PERIODOS, ARCHIVOS_CSV):
    muestra = pl.read_csv(csv, separator=SEPARADORES[periodo], n_rows=5, infer_schema=False)
    muestra = muestra.rename({c: c.lstrip('﻿') for c in muestra.columns})
    assert muestra.width == 38, f'{periodo}: {muestra.width} columnas'
    assert all(tipo == pl.String for tipo in muestra.dtypes), f'{periodo}: columnas no texto'
    if COLUMNAS_ORIGINALES is None:
        COLUMNAS_ORIGINALES = muestra.columns
    assert muestra.columns == COLUMNAS_ORIGINALES, f'{periodo}: contrato de columnas distinto'
    print(f'{csv.name} · separador {SEPARADORES[periodo]!r} · {muestra.width} columnas texto')
    display(muestra.select(['periodo', 'mes', 'afec_edad', 'afec_cod_depto',
                            'cod_motesp', 'patologia_1', 'pqr_canal']))
COLUMNAS_PARQUET = COLUMNAS_ORIGINALES + COLUMNAS_DERIVADAS


# 2.3 Plan lazy de normalización por corte (scan_csv + with_columns)

def plan_normalizado(periodo: str, csv: Path) -> pl.LazyFrame:
    lf = pl.scan_csv(csv, separator=SEPARADORES[periodo], infer_schema=False)
    con_bom = {c: c.lstrip('﻿') for c in lf.collect_schema().names() if c.startswith('﻿')}
    if con_bom:
        lf = lf.rename(con_bom)

    depto_txt = pl.col('afec_cod_depto').str.strip_chars()
    depto_2d = pl.when(depto_txt.str.contains(r'^\d{1,2}$')).then(depto_txt.str.zfill(2))
    depto_valido = depto_2d.is_in(list(DEPARTAMENTOS)).fill_null(False)

    edad_num = (pl.col('afec_edad').str.strip_chars()
                .str.replace(',', '.', literal=True).cast(pl.Float64, strict=False))
    edad_valida = (edad_num.is_finite() & (edad_num >= 0)
                   & (edad_num <= EDAD_MAXIMA)).fill_null(False)

    anio_num = pl.col('periodo').str.strip_chars().cast(pl.Int32, strict=False)
    mes_txt = pl.col('mes').str.strip_chars().str.to_lowercase().replace(MESES_TEXTO)
    mes_num = mes_txt.cast(pl.Int32, strict=False)
    fecha_valida = (anio_num.is_between(ANIO_MINIMO, ANIO_MAXIMO)
                    & mes_num.is_between(1, 12)).fill_null(False)
    anio_corte, meses_corte = MESES_CORTE[periodo]

    motivo_txt = pl.col('cod_motesp').str.strip_chars()
    return (lf.with_columns(
                pl.lit(periodo).alias('corte_archivo'),
                pl.col('cod_motesp').alias('cod_motesp_origen'),
                pl.when(motivo_txt.str.contains(rf'^\d{{1,{ANCHO_COD_MOTESP}}}$'))
                  .then(motivo_txt.str.zfill(ANCHO_COD_MOTESP))
                  .otherwise(motivo_txt).alias('cod_motesp'),
                pl.when(depto_valido).then(depto_2d).alias('codigo_departamento'),
                depto_valido.alias('departamento_valido'),
                pl.when(edad_valida).then(edad_num.floor()).cast(pl.Int16).alias('edad_anios'),
                edad_valida.alias('edad_valida'),
                pl.when(fecha_valida).then(anio_num).cast(pl.Int16).alias('anio'),
                pl.when(fecha_valida).then(mes_num).cast(pl.Int8).alias('mes_num'),
                fecha_valida.alias('fecha_valida'),
                ((anio_num == anio_corte) & mes_num.is_in(meses_corte))
                  .fill_null(False).alias('fecha_en_corte'))
            .select(COLUMNAS_PARQUET))


# 2.4 Verificación de Parquet existentes antes de reutilizarlos
═══════════════════════════════════
def firma_csv(ruta: Path) -> dict:
    estado = ruta.stat()
    return {'archivo': ruta.name, 'bytes': estado.st_size, 'mtime_ns': estado.st_mtime_ns}

def revisar_parquet(periodo: str, csv: Path, parquet: Path, filas: int) -> tuple[bool, str]:
    manifiesto = parquet.with_suffix('.json')
    if not (parquet.is_file() and manifiesto.is_file()):
        return False, 'no existe Parquet o manifiesto'
    try:
        info = json.loads(manifiesto.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return False, 'manifiesto ilegible'
    if info.get('csv') != firma_csv(csv):
        return False, 'el CSV de origen cambió (tamaño o fecha)'
    if info.get('version_reglas') != VERSION_REGLAS or info.get('corte') != periodo:
        return False, 'cambiaron las reglas de normalización'
    try:
        esquema = pl.scan_parquet(parquet).collect_schema()
        n = pl.scan_parquet(parquet).select(pl.len()).collect().item()
    except Exception as error:
        return False, f'Parquet ilegible: {error}'
    if esquema.names() != COLUMNAS_PARQUET:
        return False, 'contrato de columnas distinto'
    if any(esquema[c] != pl.String for c in COLUMNAS_ORIGINALES):
        return False, 'alguna columna original dejó de ser texto'
    if n != filas:
        return False, f'{n:,} filas; se esperaban {filas:,}'
    return True, 'Parquet verificado'

def escribir_parquet(plan: pl.LazyFrame, destino: Path) -> None:
    temporal = destino.parent / (destino.name + '.tmp')
    try:  # Sink en streaming CPU, Zstandard
        plan.sink_parquet(temporal, compression='zstd', engine='streaming')
    except TypeError:  # Polars sin parámetro engine en sinks: el sink ya es streaming
        plan.sink_parquet(temporal, compression='zstd')
    temporal.replace(destino)  # renombrado atómico: nunca queda un Parquet a medias

# ════════════════════════════════════════════════════════════════════════════
# 2.5 Conversión (CPU) con tiempo propio, separado del benchmark
# ════════════════════════════════════════════════════════════════════════════
conversion = []
for periodo, csv, parquet, filas in zip(PERIODOS, ARCHIVOS_CSV, ARCHIVOS_PARQUET, FILAS_ESPERADAS):
    reutilizable, motivo = revisar_parquet(periodo, csv, parquet, filas)
    segundos = 0.0
    if not reutilizable:
        print(f'{periodo}: se convierte ({motivo})')
        inicio = perf_counter()
        escribir_parquet(plan_normalizado(periodo, csv), parquet)
        segundos = perf_counter() - inicio
        parquet.with_suffix('.json').write_text(json.dumps({
            'corte': periodo, 'csv': firma_csv(csv), 'version_reglas': VERSION_REGLAS,
            'separador': SEPARADORES[periodo], 'polars': pl.__version__,
            'segundos_conversion': segundos}, indent=2), encoding='utf-8')
        reutilizable, motivo = revisar_parquet(periodo, csv, parquet, filas)
        assert reutilizable, f'{periodo}: conversión no verificada ({motivo})'
        accion = 'convertido'
    else:
        accion = 'reutilizado'
    conversion.append({'corte': periodo, 'accion': accion, 'control': motivo,
                       'segundos_conversion': round(segundos, 3), 'filas': filas,
                       'mb_csv': round(csv.stat().st_size / 2**20, 1),
                       'mb_parquet': round(parquet.stat().st_size / 2**20, 1)})
conversion = pl.DataFrame(conversion)
conversion.write_csv(OUT / 'conversion_PD06.csv')
display(conversion)

# ════════════════════════════════════════════════════════════════════════════
# 2.6 Calidad por corte y coherencia código ↔ descripción del motivo
# ════════════════════════════════════════════════════════════════════════════
calidad = (pl.scan_parquet(ARCHIVOS_PARQUET)
           .group_by('corte_archivo')
           .agg(pl.len().alias('reportes'),
                (pl.col('afec_cod_depto').str.strip_chars().str.len_chars() == 1).sum()
                  .alias('depto_sin_cero_origen'),
                (~pl.col('departamento_valido')).sum().alias('depto_invalido'),
                (~pl.col('edad_valida')).sum().alias('edad_nula'),
                (~pl.col('fecha_valida')).sum().alias('fecha_invalida'),
                (~pl.col('fecha_en_corte')).sum().alias('fecha_fuera_de_corte'),
                (pl.col('cod_motesp') != pl.col('cod_motesp_origen')).sum()
                  .alias('cod_motesp_rellenado'),
                pl.col('codigo_departamento').drop_nulls().n_unique().alias('departamentos'),
                pl.col('edad_anios').min().alias('edad_min'),
                pl.col('edad_anios').max().alias('edad_max'))
           .sort('corte_archivo').collect())
display(calidad)
calidad.write_csv(OUT / 'calidad_por_corte_PD06.csv')

mapa_motivos = (pl.scan_parquet(ARCHIVOS_PARQUET)
                .group_by('cod_motesp')
                .agg(pl.col('motivo_especifico').n_unique().alias('descripciones'),
                     pl.col('corte_archivo').n_unique().alias('cortes'))
                .collect())
assert (mapa_motivos['descripciones'] == 1).all(), 'Un código de motivo con varias descripciones'
print(f'{mapa_motivos.height} códigos de motivo específico; cada uno con una sola descripción.')


raise NotImplementedError('Desarrollar lectura, normalización y conversión con Polars')


PQRD_2023_II.csv · separador ';' · 38 columnas texto


periodo,mes,afec_edad,afec_cod_depto,cod_motesp,patologia_1,pqr_canal
str,str,str,str,str,str,str
"""2023""","""7""","""69""","""11""","""010103""","""ENFERMEDADES CRÓNICAS NO TRASM…","""Personalizado"""
"""2023""","""7""","""35""","""11""","""040402""","""ENFERMEDADES HUÉRFANAS""","""Web"""
"""2023""","""7""","""77""","""70""","""010301""","""ENFERMEDADES CRÓNICAS NO TRASM…","""Personalizado"""
"""2023""","""7""","""71""","""76""","""010104""","""PROBLEMAS RELACIONADOS CON FAC…","""Escrito"""
"""2023""","""7""","""48""","""08""","""010304""","""PROBLEMAS RELACIONADOS CON FAC…","""Personalizado"""


PQRD_2024_I.csv · separador ',' · 38 columnas texto


periodo,mes,afec_edad,afec_cod_depto,cod_motesp,patologia_1,pqr_canal
str,str,str,str,str,str,str
"""2024""","""6""","""73""","""5""","""10304""","""ENFERMEDADES OSTEOARTICULARES""","""Telefónico"""
"""2024""","""4""","""84""","""63""","""10204""","""PROBLEMAS RELACIONADOS CON FAC…","""Redes Sociales"""
"""2024""","""5""","""0""","""11""","""10103""","""MATERNO INFANTIL""","""Web"""
"""2024""","""6""","""62""","""11""","""10301""","""ENFERMEDADES OSTEOARTICULARES""","""Web"""
"""2024""","""6""","""39""","""66""","""10103""","""CÁNCER ""","""Personalizado"""


PQRD_2024_II.csv · separador ';' · 38 columnas texto


periodo,mes,afec_edad,afec_cod_depto,cod_motesp,patologia_1,pqr_canal
str,str,str,str,str,str,str
"""2024""","""7""","""71""","""05""","""010101""","""ENFERMEDADES CRÓNICAS NO TRASM…","""Web"""
"""2024""","""7""","""61""","""11""","""010203""","""PROBLEMAS RELACIONADOS CON FAC…","""Web"""
"""2024""","""7""","""32""","""66""","""010101""","""ENFERMEDADES CRÓNICAS NO TRANS…","""Web"""
"""2024""","""7""","""9""","""66""","""010101""","""PROBLEMAS RELACIONADOS CON FAC…","""Web"""
"""2024""","""7""","""33""","""05""","""010103""","""PROBLEMAS RELACIONADOS CON FAC…","""Telefónico"""


2023_II: se convierte (no existe Parquet o manifiesto)
2024_I: se convierte (no existe Parquet o manifiesto)
2024_II: se convierte (no existe Parquet o manifiesto)


corte,accion,control,segundos_conversion,filas,mb_csv,mb_parquet
str,str,str,f64,i64,f64,f64
"""2023_II""","""convertido""","""Parquet verificado""",4.498,716697,508.0,16.4
"""2024_I""","""convertido""","""Parquet verificado""",3.758,781601,550.0,18.1
"""2024_II""","""convertido""","""Parquet verificado""",6.495,946468,674.1,21.4


corte_archivo,reportes,depto_sin_cero_origen,depto_invalido,edad_nula,fecha_invalida,fecha_fuera_de_corte,cod_motesp_rellenado,departamentos,edad_min,edad_max
str,u32,u32,u32,u32,u32,u32,u32,u32,i16,i16
"""2023_II""",716697,0,0,0,0,0,0,33,0,113
"""2024_I""",781601,167477,0,1,0,0,781601,33,0,113
"""2024_II""",946468,0,0,0,0,0,0,33,0,118


37 códigos de motivo específico; cada uno con una sola descripción.


NotImplementedError: Desarrollar lectura, normalización y conversión con Polars

In [5]:
# Control de preparación; ejecutar después de completar la celda anterior.
assert all(p.is_file() for p in ARCHIVOS_PARQUET), 'Faltan periodos convertidos'
filas_por_archivo = [pl.scan_parquet(p).select(pl.len()).collect().item()
                     for p in ARCHIVOS_PARQUET]
assert filas_por_archivo == FILAS_ESPERADAS
assert sum(filas_por_archivo) == TOTAL_ESPERADO
REQUERIDAS = ['codigo_departamento', 'edad_anios', 'patologia_1',
              'cod_motesp', 'motivo_especifico', 'pqr_canal', 'anio', 'mes_num']
for archivo in ARCHIVOS_PARQUET:
    esquema = pl.scan_parquet(archivo).collect_schema()
    assert set(REQUERIDAS) <= set(esquema.names())
    assert esquema['anio'].is_integer() and esquema['mes_num'].is_integer()
    assert esquema['edad_anios'].is_numeric()
print('Tres periodos conservados:', filas_por_archivo)


AssertionError: Faltan periodos convertidos

## 3. Ejercicio 1 · Departamentos con más quejas por perfil

**Combina todos los periodos completos**, no solo una muestra. Define el perfil:

| Dimensión | Campo |
|---|---|
| Edad en años | `edad_anios`, construida desde `afec_edad` |
| Patología | `patologia_1` |
| Código del motivo específico | `cod_motesp` |
| Descripción del mismo motivo | `motivo_especifico` |
| Canal de entrada | `pqr_canal` |

No mezcles un código de motivo específico con la descripción de otro nivel.
Agrupa por departamento y las cinco dimensiones; cuenta filas como `quejas`.
Ordena por perfil, quejas descendentes y código de departamento ascendente
para resolver empates de forma reproducible. Conserva los grupos no informados.

**Producto:** una tabla detallada con todas las combinaciones y sus conteos;
y una vista de hasta `TOP_N` departamentos con más quejas **para cada perfil**.
No basta un único top nacional que ignore edad, patología, motivo y canal.
Puedes preparar ese recorte después de medir, indicándolo en el informe.

El plan medido debe incluir lectura Parquet, agrupación y ordenamiento.
Declara un `LazyFrame` llamado `consulta_e1`; no lo ejecutes todavía.
No presentes quejas como personas únicas ni como tasas por población.


In [ ]:
# TODO: desarrollar las expresiones y la consulta LazyFrame del ejercicio 1.
# Deben verse select, group_by, agg y sort.
consulta_e1 = None
raise NotImplementedError('Desarrollar la agregación y ordenación del ejercicio 1')


## 4. Ejercicio 2 · Evolución mensual de los mismos perfiles

Conserva departamento, edad, patología, código y descripción del motivo y canal.
Añade **año y mes numéricos** a la agrupación, cuenta las quejas y ordena el
resultado **primero por año y mes ascendentes**, después por las dimensiones
para que la salida sea reproducible. Enero, febrero y octubre deben quedar
como meses 1, 2 y 10, no como cadenas ordenadas alfabéticamente.

**Producto:** tabla `anio`, `mes_num`, departamento, las cinco dimensiones y
`quejas`, que permita seguir un mismo perfil mes a mes. Muestra dos perfiles
concretos y explica sus cambios. No concluyas que toda variación es tendencia.

Conserva las fechas inválidas en el control total y sepáralas al presentar la
cronología. No inventes meses ni rellenes silenciosamente ausencias con cero.
Declara el plan lazy como `consulta_e2`, sin ejecutarlo todavía.


In [ ]:
# TODO: desarrollar el ejercicio 2: mismas dimensiones + anio y mes_num.
# La ordenación por anio y mes_num debe estar en el plan medido.
consulta_e2 = None
raise NotImplementedError('Desarrollar la agregación mensual y su orden ascendente')


## 5. Contrato de comparación y frontera temporal

Completa las listas de claves y la comprobación para cada resultado.
Verifica todas las filas agregadas, no solo la suma global. Los conteos deben
ser exactos y cada consulta debe conservar 2.444.766 reportes, incluidos grupos
no informados. Revisa tipos, nulos y duplicados de claves agregadas.

Mediremos con `perf_counter`: **leer Parquet + agregar + ordenar + devolver un
DataFrame Polars en CPU**. `collect` termina esa frontera para ambos motores.
No se mide el tiempo de construir el plan, descargar, convertir, verificar ni
exportar. La medición GPU incluye transferencias; no es tiempo exclusivo de
kernels CUDA ni una comparación de datos ya residentes en VRAM.

CPU usará `engine='streaming'`. GPU usará `GPUEngine(raise_on_fail=True)`.
Si el plan no está soportado en GPU, corrígelo o informa el error; no lo midas
como GPU mediante fallback a CPU. No cambies filtros o claves entre motores.


In [ ]:
assert isinstance(consulta_e1, pl.LazyFrame), 'Completar ejercicio 1'
assert isinstance(consulta_e2, pl.LazyFrame), 'Completar ejercicio 2'
CONSULTAS = {'e1_departamentos': consulta_e1, 'e2_evolucion_mensual': consulta_e2}
CLAVES_RESULTADO = {
    'e1_departamentos': [],  # TODO: claves completas, sin la columna quejas
    'e2_evolucion_mensual': []
}
assert all(CLAVES_RESULTADO.values()), 'Definir las claves para comparar todos los grupos'
engine_gpu = pl.GPUEngine(device=DISPOSITIVO_CUDA, raise_on_fail=True)

# TODO: comprobar esquema, unicidad de claves, nulos y total de cada resultado.
# Mantén visibles las comprobaciones en el notebook antes de interpretar tiempos.
primeras = []


### Ejercicio 1 · Primera ejecución en CPU y GPU

Ejecuta la celda después de completar los planes. La primera llamada puede
incluir inicialización y cachés; se guarda por separado de las repeticiones.


In [ ]:
# CPU: se cronometra la ejecución completa del plan, no su declaración.
inicio = perf_counter()
cpu_e1 = consulta_e1.collect(engine='streaming')
tiempo_cpu_e1 = perf_counter() - inicio

# GPU: mismos datos, claves, agregación y orden.
inicio = perf_counter()
gpu_e1 = consulta_e1.collect(engine=engine_gpu)
tiempo_gpu_e1 = perf_counter() - inicio

# Fuera del reloj: comparar todos los grupos y conservar el total.
assert_frame_equal(cpu_e1.sort(CLAVES_RESULTADO['e1_departamentos']),
                   gpu_e1.sort(CLAVES_RESULTADO['e1_departamentos']), check_exact=True)
assert cpu_e1['quejas'].sum() == TOTAL_ESPERADO
primeras.extend([
    {'ejercicio': 'e1', 'motor': 'cpu', 'segundos': tiempo_cpu_e1},
    {'ejercicio': 'e1', 'motor': 'gpu', 'segundos': tiempo_gpu_e1}])
print('Ejercicio 1 · CPU:', tiempo_cpu_e1, 's · GPU:', tiempo_gpu_e1, 's')
display(gpu_e1.head(20))


### Ejercicio 2 · Primera ejecución en CPU y GPU

Ejecuta la celda después de completar los planes. La primera llamada puede
incluir inicialización y cachés; se guarda por separado de las repeticiones.


In [ ]:
# CPU: se cronometra la ejecución completa del plan, no su declaración.
inicio = perf_counter()
cpu_e2 = consulta_e2.collect(engine='streaming')
tiempo_cpu_e2 = perf_counter() - inicio

# GPU: mismos datos, claves, agregación y orden.
inicio = perf_counter()
gpu_e2 = consulta_e2.collect(engine=engine_gpu)
tiempo_gpu_e2 = perf_counter() - inicio

# Fuera del reloj: comparar todos los grupos y conservar el total.
assert_frame_equal(cpu_e2.sort(CLAVES_RESULTADO['e2_evolucion_mensual']),
                   gpu_e2.sort(CLAVES_RESULTADO['e2_evolucion_mensual']), check_exact=True)
assert cpu_e2['quejas'].sum() == TOTAL_ESPERADO
primeras.extend([
    {'ejercicio': 'e2', 'motor': 'cpu', 'segundos': tiempo_cpu_e2},
    {'ejercicio': 'e2', 'motor': 'gpu', 'segundos': tiempo_gpu_e2}])
print('Ejercicio 2 · CPU:', tiempo_cpu_e2, 's · GPU:', tiempo_gpu_e2, 's')
display(gpu_e2.head(20))


## 6. Calentamiento y tres repeticiones por motor

Completa el bucle de medición. Por cada ejercicio:

1. Ejecuta una vuelta adicional de calentamiento CPU/GPU y no la mezcles con las medidas.
2. Repite al menos tres veces, alternando CPU→GPU y GPU→CPU.
3. Rodea cada `collect` con `perf_counter`; compara resultados después de parar el reloj.
4. Guarda `ejercicio`, `motor`, `repeticion`, `segundos`, `filas_entrada`,
   `grupos_resultado` y `equivalente` en la lista `mediciones`.
5. Revisa que los resultados conserven todos los reportes y las claves esperadas.

Mide los mismos planes usados arriba. Mantén las mismas condiciones y registra
hilos CPU, GPU, VRAM, controlador y versiones. No se borran automáticamente las
cachés del sistema operativo; explica su efecto. No uses `process_time` como
sustituto de la duración total de la consulta GPU.


In [ ]:
mediciones = []
# TODO: desarrollar aquí calentamiento, alternancia, reloj, validación y registro.
# La ejecución de cada collect debe quedar visible en este bucle.
raise NotImplementedError('Completar las tres repeticiones CPU/GPU de ambos ejercicios')


## 7. Tabla de tiempos y factor de aceleración

La celda siguiente no resuelve consultas: resume tus mediciones. Habrá
**12 filas como mínimo**: dos ejercicios × dos motores × tres repeticiones.
Factor CPU/GPU = mediana CPU / mediana GPU: mayor que 1 favorece GPU;
menor que 1 favorece CPU. No mezcles ambas consultas ni las primeras ejecuciones.


In [ ]:
assert len(mediciones) == len(CONSULTAS) * 2 * REPETICIONES
requeridas = {'ejercicio', 'motor', 'repeticion', 'segundos',
              'filas_entrada', 'grupos_resultado', 'equivalente'}
assert all(requeridas <= set(fila) for fila in mediciones)
tiempos = pl.DataFrame(mediciones)
assert tiempos['equivalente'].all()
assert (tiempos['filas_entrada'] == TOTAL_ESPERADO).all()
assert tiempos['segundos'].is_finite().all() and (tiempos['segundos'] > 0).all()
assert set(tiempos['ejercicio'].to_list()) == set(CONSULTAS)
assert set(tiempos['motor'].to_list()) == {'cpu', 'gpu'}
conteos = tiempos.group_by(['ejercicio', 'motor']).agg(pl.len().alias('n'))
assert (conteos['n'] == REPETICIONES).all()
assert tiempos.select(['ejercicio', 'motor', 'repeticion']).unique().height == tiempos.height
resumen = tiempos.group_by(['ejercicio', 'motor']).agg(
    pl.col('segundos').median().alias('mediana_s'),
    pl.col('segundos').min().alias('min_s'),
    pl.col('segundos').max().alias('max_s')).sort(['ejercicio', 'motor'])
factores = []
for nombre in CONSULTAS:
    cpu_s = resumen.filter((pl.col('ejercicio') == nombre) & (pl.col('motor') == 'cpu'))['mediana_s'].item()
    gpu_s = resumen.filter((pl.col('ejercicio') == nombre) & (pl.col('motor') == 'gpu'))['mediana_s'].item()
    factores.append({'ejercicio': nombre, 'mediana_cpu_s': cpu_s,
                     'mediana_gpu_s': gpu_s, 'factor_cpu_sobre_gpu': cpu_s / gpu_s})
display(resumen)
display(pl.DataFrame(factores))
tiempos.write_csv(OUT / 'benchmark_PD06.csv')
resumen.write_csv(OUT / 'resumen_PD06.csv')
pl.DataFrame(primeras).write_csv(OUT / 'primeras_ejecuciones_PD06.csv')
pl.DataFrame(factores).write_csv(OUT / 'factores_PD06.csv')


## 8. Interpretar y entregar

Completa el notebook y entrega en tu repositorio:

- `soluciones/PD06_PQRS_agregaciones_CPU_GPU.ipynb` con código visible y conclusiones.
- Resultados regenerables en `kit/salidas/soluciones/PD06/`: detalle/ranking de
  departamentos, evolución mensual, controles, tiempos, resumen y factores.
- Un control JSON con filas por corte, total conciliado, número de grupos por
  consulta, reglas de edad y fechas, cobertura de departamentos, versiones,
  modelo GPU/VRAM/controlador, hilos CPU y alcance de cada medición.

**Responde:** ¿qué departamentos lideran dentro de cada perfil? ¿Cómo evolucionan
los dos perfiles seleccionados? ¿Cuántos grupos genera cada consulta? ¿La GPU
acelera ambas? ¿Qué parte puede estar limitada por lectura, transferencias,
ordenamiento o memoria? ¿Cambió la conclusión entre primera ejecución y mediana?

Se evalúan corrección, equivalencia, medición reproducible e interpretación;
no que GPU necesariamente gane. No adjuntes los CSV originales al repositorio.
La salida queda en la ruta ignorada `kit/salidas/`. La copia resuelta pertenece
al estudiante; esta plantilla mantiene sus celdas por completar.

Apoyo: [guía Polars](../docs/manual-polars.md),
[entorno GPU](../docs/polars-gpu.md),
[referencia P04](../talleres/P04.md). Son referencias, no una solución de PD06.
